# Thermal-Gradient Modeling of Large-Scale Structures
## Karla Peña Ramirez

The goal of this analysis is the evaluation of the safety limits on M1M3 transport (single 8.4-meter monolithic borosilicate glass substrat) to the dome and its temperature once it is installed at the Simonyi Telescope. Moving such a large structure is not only a mechanical challenge but a thermal one: we need less than 5 °C temperature variation in the glass surface and the temperature in the mirror can not change more than 1 °C per hour.

In [ ]:
#%load_ext lab_black
#%load_ext autoreload
#%autoreload 2

In [ ]:
# Preliminars.
import asyncio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import sys
import time
import warnings
import seaborn as sns
import datetime as dt

from astropy.time import Time, TimeDelta
from lsst.summit.utils.efdUtils import makeEfdClient
from lsst_efd_client.efd_helper import merge_packed_time_series
from lsst_efd_client import EfdClient

# Ignore the many warning messages from ``merge_packed_time_series``.
warnings.simplefilter(action="ignore", category=FutureWarning)

client = makeEfdClient()
#client = EfdClient()

We are interested in retrieving temperature information from two sensors: ESS:103 (MTDome) and ESS:301 (Weather tower).

In [ ]:
# Data retrieval in the time range of interest.
start = Time("2024-08-17 00:00:00Z", scale="utc")
end = Time("2024-10-02 18:00:00Z", scale="utc")

#Data exploration on temperature data.
packed_dataframe = await client.select_time_series(
    "lsst.sal.ESS.temperature", ["*"], start, end
)

Let's first make a data exploration. Let's check out the available sensor and the number of data points we have at our disposal for the analysis.

In [ ]:
# Data exploration.
packed_dataframe.columns
packed_dataframe.sensorName.value_counts()
packed_dataframe.private_identity.value_counts()

There are eight sensors. Let's isolate data on the MTDome sensor ESS:103.

In [ ]:
#MTDome sensor.
ESS_103 = packed_dataframe[packed_dataframe["private_identity"] == 'ESS:103']

Let's see the data. The temperatureItem0 is the column that gathers the information we are requiring.

In [ ]:
#MTDome sensor.
ESS_103.columns
ESS_103
ESS_103.temperatureItem7.describe()
ESS_103.temperatureItem6.describe()
ESS_103.temperatureItem5.describe()
ESS_103.temperatureItem4.describe()
ESS_103.temperatureItem3.describe()
ESS_103.temperatureItem2.describe()
ESS_103.temperatureItem1.describe()
ESS_103.temperatureItem0.describe()

Similarly we have the temperature data from the Weather tower (ESS:301)

In [ ]:
#Weather tower sensor.
ESS_301 = packed_dataframe[packed_dataframe["private_identity"] == 'ESS:301']

## Temperature comparison between MTDome and Weather tower sensors.

The M1M3 safety limit is 5°C variations over the mirror surface (and thus the environment to ensure the glass safety).

In [ ]:
#Raw draft figure.
#plt.figure(figsize = (6,4))
plt.title('Sensor Set 1')
ESS_103['temperatureItem0'].plot(linewidth=1)
ESS_301['temperatureItem0'].plot(linewidth=1)
plt.legend(['MTDome','Weather tower'])
plt.ylabel('Temperature [∘C]')
plt.show()


In [ ]:
#Difference in temperature per hour between the two locations. Raw draft figure.
D103 = ESS_103.resample(rule='60min').apply({'temperatureItem0': ['mean','min','max']})
D301 = ESS_301.resample(rule='60min').apply({'temperatureItem0': ['mean','min','max']})
delta = D103 - D301
delta = delta.dropna()

fig, ax = plt.subplots()

delta.plot(linewidth=1, legend=False)
#ax.fill_between(delta.index,  delta['max'],  delta['min'], alpha=.2)
plt.title('Sensor Set 1: MTDome - Weather Tower')
plt.ylabel('Temperature [∘C]')
plt.axhline(y=5.0, color='r', linestyle='--', alpha=0.4)
plt.axhline(y=-5.0, color='r', linestyle='--', alpha=0.4)
plt.show()

In [ ]:
#Raw draft figure. Resampling 6H.
D103 = ESS_103['temperatureItem0'].resample("6H").mean()
D301 = ESS_301['temperatureItem0'].resample("6H").mean()
#plt.figure(figsize = (6,4))
plt.title('Sensor Set 1')
D103.plot(linewidth=1)
D301.plot(linewidth=1)
plt.legend(['MTDome','Weather tower'])
plt.ylabel('Temperature [∘C]')
plt.show()


####  Final plots

In [ ]:
#Figure 1. Raw data
ESS_103.temperatureItem0.plot(linewidth=1, label='MTDome')
ESS_301.temperatureItem0.plot(linewidth=1, label='Weather tower')
#ESS_103.temperatureItem0.rolling('12H').mean().plot(label='12 hour mean')
#ESS_103.temperatureItem0.expanding().mean().plot(label='Expanding Average')
#ESS_103.temperatureItem0.ewm(alpha=0.1).mean().plot(label='EWMA($\\alpha=0.1$)')

plt.title('Sensor Set 1')
plt.legend()
plt.ylabel("Temperature [∘C]")
sns.despine()
plt.savefig('Sensor1_raw.png', dpi=300)

In [ ]:
#Figure 2. Resample every 6H.
roll_103 = ESS_103.temperatureItem0.rolling('6H', center=True)
m1 = roll_103.agg(['mean', 'min', 'max'])

roll_301 = ESS_301.temperatureItem0.rolling('6H', center=True)
m2 = roll_301.agg(['mean', 'min', 'max'])

plt.title('Sensor Set 1: 6 hour resampling')
ax = m1['mean'].plot(label='MTDome', linewidth=1)
ax.fill_between(m1.index,  m1['max'],  m1['min'], alpha=.3)
ax = m2['mean'].plot(label='Weather tower', linewidth=1)
ax.fill_between(m2.index,  m2['max'],  m2['min'], alpha=.3)
plt.ylabel("Temperature [∘C]")
plt.legend()
sns.despine()
plt.savefig('Sensor1_6h_raw.png', dpi=300)

In [ ]:
#Figure 3. Difference in temperature per hour between the two locations.
mean_dome = ESS_103.resample(rule='1H').apply({'temperatureItem0': ['mean','min','max']})
mean_tower = ESS_301.resample(rule='1H').apply({'temperatureItem0': ['mean','min','max']})
delta = mean_dome - mean_tower
delta = delta.dropna()

ax = delta.temperatureItem0['mean'].plot(legend=False,linewidth=1)
ax.fill_between(delta.index,  delta.temperatureItem0['min'],  delta.temperatureItem0['max'], alpha=.5)

plt.title('Sensor Set 1: MTDome - Weather Tower')
plt.ylabel("Temperature [∘C]")
plt.axhline(y=5.0, color='r', linestyle='--', alpha=0.3)
plt.axhline(y=-5.0, color='r', linestyle='--', alpha=0.3)
sns.despine()
plt.savefig('Sensor1_1h_delta_temp.png', dpi=300)



## MTDome temperature change within 1 hour

We need information about the change in temperature inside the dome each hour. Therefore, first we resample the data each 60 minutes and then make the derivate of the values. We are interested in the mean, min and max variations.

In [ ]:
#Figure 4. Resample every 1h and derivative.
# Data resampling and data derivative on the mean.
H103 = ESS_103.resample(rule='60min').apply({ 
                'temperatureItem0': ['mean','min','max']}).diff()

H103 = H103.dropna()
ax = H103.temperatureItem0['mean'].plot(linewidth=1, legend=False)
ax.fill_between(H103.index,  H103.temperatureItem0['min'],  H103.temperatureItem0['max'], alpha=.9)

plt.title('Sensor Set 1: Rate of temperature change per hour')
plt.ylabel('Rate of temperature change [∘C/hr]')
plt.axhline(y=1.0, color='r', linestyle='--', alpha=0.3)
plt.axhline(y=-1.0, color='r', linestyle='--', alpha=0.3)
sns.despine()
plt.savefig('Sensor1_1h_temp_derivative.png', dpi=300)

First event: When resampling the data and focusing on the minimum temperature values we see changes close to 1°C/hr. Values below 1°C/hr can be handled when the M1M3 thermal system is on. Let's take a closer look on that event.

In [ ]:
# Data retrieval in the time range of interest.
start = Time("2024-08-20 03:00:00Z", scale="utc")
end = Time("2024-08-20 21:00:00Z", scale="utc")

#Data exploration on temperature data.
packed_dataframe = await client.select_time_series(
    "lsst.sal.ESS.temperature", ["*"], start, end
)

In [ ]:
#MTDome sensor.
ESS_103e1 = packed_dataframe[packed_dataframe["private_identity"] == 'ESS:103']
#Weather tower sensor.
ESS_301e1 = packed_dataframe[packed_dataframe["private_identity"] == 'ESS:301']

In [ ]:
print(ESS_103e1.dtypes)
ESS_103e1

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=1)

ESS_103e1['temperatureItem0'].plot()
ESS_301e1['temperatureItem0'].plot()
plt.legend(['MTDome','Weather tower'])
plt.title('Sensor comparison')
plt.ylabel('Temperature [∘C]')
sns.despine()
plt.savefig('Sensor1_raw_e1.png', dpi=300)

In [ ]:
# Data resampling and data derivative on the min.
H103e1 = ESS_103e1.resample(rule='60min').apply({ 
                'temperatureItem0': ['mean','min','max']}).diff()

H103e1 = H103e1.dropna()
ax = H103e1.temperatureItem0['mean'].plot(linewidth=1, legend=False)
ax.fill_between(H103e1.index,  H103e1.temperatureItem0['min'],  H103e1.temperatureItem0['max'], alpha=.2)

plt.title('Sensor Set 1: Rate of temperature change per hour')
plt.ylabel('Rate of temperature change [∘C/hr]')
plt.axhline(y=1.0, color='r', linestyle='--', alpha=0.3)
plt.axhline(y=-1.0, color='r', linestyle='--', alpha=0.3)
sns.despine()
plt.savefig('Sensor1_1h_temp_derivative_e1.png', dpi=300)

Second event: When resampling the data and focusing on the minimum temperature values we see changes close to 1°C/hr. Values below 1°C/hr can be handled when the M1M3 thermal system is on. Let's take a closer look on that event.

In [ ]:
# Data retrieval in the time range of interest.
start = Time("2024-09-25 06:00:00Z", scale="utc")
end = Time("2024-09-25 21:00:00Z", scale="utc")

#Data exploration on temperature data.
packed_dataframe = await client.select_time_series(
    "lsst.sal.ESS.temperature", ["*"], start, end
)

In [ ]:
#MTDome sensor.
ESS_103e2 = packed_dataframe[packed_dataframe["private_identity"] == 'ESS:103']
#Weather tower sensor.
ESS_301e2 = packed_dataframe[packed_dataframe["private_identity"] == 'ESS:301']

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=1)

ESS_103e2['temperatureItem0'].plot()
ESS_301e2['temperatureItem0'].plot()
plt.legend(['MTDome','Weather tower'])
plt.title('Sensor comparison')
plt.ylabel('Temperature [∘C]')
sns.despine()
plt.savefig('Sensor1_raw_e2.png', dpi=300)

In [ ]:
# Data resampling and data derivative on the min.
H103e2 = ESS_103e2.resample(rule='60min').apply({ 
                'temperatureItem0': ['mean','min','max']}).diff()

H103e2 = H103e2.dropna()
ax = H103e2.temperatureItem0['mean'].plot(linewidth=1, legend=False)
ax.fill_between(H103e2.index,  H103e2.temperatureItem0['min'],  H103e2.temperatureItem0['max'], alpha=.2)

plt.title('Sensor Set 1: Rate of temperature change per hour')
plt.ylabel('Rate of temperature change [∘C/hr]')
plt.axhline(y=1.0, color='r', linestyle='--', alpha=0.3)
plt.axhline(y=-1.0, color='r', linestyle='--', alpha=0.3)
sns.despine()
plt.savefig('Sensor1_1h_temp_derivative_e2.png', dpi=300)

Now lets see the day/night trend to get the best moment in the day to make the transportation.

In [ ]:
# Data retrieval in the time range of interest.
start = Time("2024-09-27 06:00:00Z", scale="utc")
end = Time("2024-10-02 21:00:00Z", scale="utc")

#Data exploration on temperature data.
packed_dataframe = await client.select_time_series(
    "lsst.sal.ESS.temperature", ["*"], start, end
)

In [ ]:
#MTDome sensor.
ESS_103t = packed_dataframe[packed_dataframe["private_identity"] == 'ESS:103']
#Weather tower sensor.
ESS_301t = packed_dataframe[packed_dataframe["private_identity"] == 'ESS:301']

In [ ]:
#Figure 3. Difference in temperature per hour between the two locations.
mean_domet = ESS_103t.resample(rule='1H').apply({'temperatureItem0': ['mean','min','max']})
mean_towert = ESS_301t.resample(rule='1H').apply({'temperatureItem0': ['mean','min','max']})
deltat = mean_domet - mean_towert
deltat = deltat.dropna()

ax = deltat.temperatureItem0['mean'].plot(legend=False,linewidth=1)
ax.fill_between(deltat.index,  deltat.temperatureItem0['min'],  deltat.temperatureItem0['max'], alpha=.5)

plt.title('Sensor Set 1: MTDome - Weather Tower')
plt.ylabel("Temperature [∘C]")
plt.axhline(y=0.0, color='k', linestyle='--', alpha=0.3, linewidth=0.5)

start = dt.datetime(2024, 9, 27, 18 ,0)
end = dt.datetime(2024, 9, 28, 8 ,0)
plt.axvspan(start, end, alpha=0.1, color='k')

start = dt.datetime(2024, 9, 28, 18 ,0)
end = dt.datetime(2024, 9, 29, 8 ,0)
plt.axvspan(start, end, alpha=0.1, color='k')

start = dt.datetime(2024, 9, 29, 18 ,0)
end = dt.datetime(2024, 9, 30, 8 ,0)
plt.axvspan(start, end, alpha=0.1, color='k')

start = dt.datetime(2024, 9, 30, 18 ,0)
end = dt.datetime(2024, 10, 1, 8 ,0)
plt.axvspan(start, end, alpha=0.1, color='k')

start = dt.datetime(2024, 10, 1, 18 ,0)
end = dt.datetime(2024, 10, 2, 8 ,0)
plt.axvspan(start, end, alpha=0.1, color='k')

start = dt.datetime(2024, 10, 2, 18 ,0)
end = dt.datetime(2024, 10, 2, 21 ,0)
plt.axvspan(start, end, alpha=0.1, color='k')

moment = dt.datetime(2024, 10, 2, 12 ,15)
plt.axvline(x=moment, color='r', linestyle='--', alpha=0.3, linewidth=1)

sns.despine()
plt.savefig('Sensor1_1h_delta_temp_moment.png', dpi=300)

All the results from the analysis are documented in the technote: https://sitcomtn-141.lsst.io/